In [118]:
import numpy as np
import scipy.sparse as sp
import time
from dolphindes.cvxopt import DenseSharedProjQCQP, OptimizationHyperparameters

# conda clean --all # in case of environment issues

In [119]:

# Physical constants
epsilon_0 = 8.854e-12 # F/m
mu_0 = 4e-7 * np.pi # H/m
c = 1/np.sqrt(epsilon_0 * mu_0) # speed of light in vacuum

frequency = 2.45e9 # frequency in Hz
wavelength = c / frequency # wavelength in m
print(f"Wavelength: {wavelength} m")

Z0 = 50 # port impedance in Ohm

ka = 1 # electrical size of the antenna
k = 2 * np.pi / wavelength # wavenumber
a = ka / k # antenna circumradius
print(f"Antenna circumradius: {a} m")

omega = 2 * np.pi * frequency # angular frequency

conductivity_reduction_factor = 1e-8 # factor to reduce conductivity for testing purposes
copper_conductivity = conductivity_reduction_factor*5.96e7 # S/m
copper_permittivity = -1j * copper_conductivity / (omega * epsilon_0) # copper permittivity in dimensionless units

Wavelength: 0.1223655664053944 m
Antenna circumradius: 0.019475084757658086 m


In [ ]:

# Calculate surface impedance
delta = np.sqrt(2 / (omega * mu_0 * copper_conductivity)) # skin depth in m
Zs = 1.0 / (copper_conductivity * delta)
print(f"Surface impedance: {np.abs(Zs)} Ω")

# Load matrices from text files
Lmat = np.loadtxt(r"Lmat_matrix.txt", delimiter=',') # inductance matrix
R0 = np.loadtxt(r"R0_matrix.txt", delimiter=',') # radiated power matrix
X0 = np.loadtxt(r"X0_matrix.txt", delimiter=',') # reactive power matrix
Vinc = np.loadtxt(r"V_vector.txt", delimiter=',') # voltage excitation vector
Rmat = Zs * Lmat # dissipation matrix
Z0 = R0 + 1j*X0 

Ndes = Lmat.shape[0] # number of design variables
print("Number of design variables: ", Ndes)



Surface impedance: 127.39130327240646 Ω
Number of design variables:  273


In [ ]:
# lossy matrix Cholesky factorization
Lchol = np.linalg.cholesky(Lmat)
Mfactor = np.linalg.inv(Lchol)

Lmat = np.eye(Ndes)
Rmat = Zs*Lmat
R0 = Mfactor.conj().T @ R0 @ Mfactor
X0 = Mfactor.conj().T @ X0 @ Mfactor
Z0 = R0 + 1j*X0
Vinc = Mfactor.conj().T @ Vinc

In [122]:
# # lossy matrix eigenvalue factorization
# [Leigvals, Mfactor] = np.linalg.eig(Lmat)
# Mfactor = Mfactor @ np.diag(1/np.sqrt(Leigvals))

# Lmat =Mfactor.conj().T @ Lmat @ Mfactor
# Rmat = Mfactor.conj().T @ Rmat @ Mfactor
# R0 = Mfactor.conj().T @ R0 @ Mfactor
# X0 = Mfactor.conj().T @ X0 @ Mfactor
# Z0 = R0 + 1j*X0
# Vinc = Mfactor.conj().T @ Vinc

In [123]:
# # rezistivity matrix Cholesky factorization
# Rchol = np.linalg.cholesky(Rmat)
# Richol = np.linalg.inv(Rchol)
 
# Rmat = np.eye(Ndes)
# R0 = Richol.conj().T @ R0 @ Richol
# X0 = Richol.conj().T @ X0 @ Richol
# Z0 = R0 + 1j*X0
# Vinc = Richol.conj().T @ Vinc

In [124]:
R0Norm = np.linalg.norm(R0, ord=2) # norm of radiated power matrix

Rmat0 = R0 + Rmat # dissipation matrix
Rmat0Norm = np.linalg.norm(Rmat0, ord=2) # norm of dissipation matrix

X0Norm = np.linalg.norm(X0, ord=2) # norm of reactive power matrix

Z0 = R0 + 1j * X0 # free space impedance matrix
Zmat = Rmat0 + 1j * X0 # impedance matrix

Vzero = np.zeros((Ndes,), dtype=complex) # zero vector
Mzero = np.zeros((Ndes,Ndes), dtype=complex) # zero matrix

# translate QCQP matrices to Dolphindes notation
Umat = 1j * Zmat.conj() # Dolphindes U matrix
eVec = -1j*Vinc.conj() / 2 # Dolphindes e vector
Bmat =  -0.5*R0 # quadratic objective matrix
bVec = Vzero/2 # linear objective vector
beta = 0 # constant objective term

In [125]:
# # preconditioning system matrix
# G0 = Z0
# alpha = -1/Zs
# X = alpha*np.eye(Ndes)
# H0 = G0 + alpha*np.eye(Ndes)
# Y = np.linalg.inv(np.eye(Ndes) + alpha*X) @ X

# S = np.eye(Ndes) - Y @ H0
# p = Y @ G0 @ (1/Zs * Vinc)

# tFull = np.linalg.solve(S, p) # full patch preconditione current distribution in preconditioned space
# IfullPrecond = tFull + (1/Zs * Vinc) # full patch preconditioned current distribution in original space
# # Vtest = Zmat @ np.linalg.solve(Zmat, Vinc) # voltage test vector
# # Vtest = Zmat @ np.linalg.solve(Zs*np.eye(Ndes) + Z0, Vinc) # voltage test vector
# Vtest = Zmat @ IfullPrecond # voltage test vector
# errVinc = np.linalg.norm(Vtest - Vinc)/np.linalg.norm(Vinc)
# print(f"preconditioned voltage error: {errVinc:.2e}")

In [126]:
# # tranformed radiated power calculation
# T = -0.5*R0
# t = - R0 @ (1/Zs *  Vinc)
# t0 = np.real((1/Zs * Vinc).conj().T @ (-0.5*R0) @ (1/Zs * Vinc))

# PradFull = 0.5 * np.real(IfullPrecond.conj().T @ R0 @ IfullPrecond)
# PradFullTransformed = - np.real(tFull.conj().T @ T @ tFull) - np.real(tFull.conj().T @ t) - t0
# errPrad = np.abs(PradFull - PradFullTransformed)/np.abs(PradFull)
# print(f"Prad error: {errPrad:.2e}")

# # translate QCQP matrices to Dolphindes notation
# Umat = 1j * S.conj() # Dolphindes U matrix
# eVec = -1j*p.conj() / 2 # Dolphindes e vector
# Bmat =  T # quadratic objective matrix
# bVec = t/2 # linear objective vector
# beta = t0 # constant objective term

In [127]:
# set up optimization variables

Pdiags = np.ones((Ndes,2), dtype=complex) # gloal constraints
Pdiags[:,1] = -1j
# norm_list = [X0Norm, Rmat0Norm]
norm_list=[1,1]
Plist = [sp.diags(Pdiags[:,i] / norm_list[i]) for i in range(Pdiags.shape[1])]

# # Set up and solve QCQP
# Pdiags = 'global' # use global constraints
QCQP = DenseSharedProjQCQP(Bmat, bVec, beta,
                            Umat, eVec,
                             Plist, verbose = 1
                                )

t1 = time.time()
lags_init = np.zeros((Pdiags.shape[1],))
lags_init[1] = 100
# lags_init = np.array([-8.53605052, 6.85531752])
# lags_init = np.array([-5.00174943e-04, 8.98932137e-01])

# New interface for specifying optimization hyperparameters. https://dolphindes.readthedocs.io/en/latest/api/dolphindes.cvxopt.OptimizationHyperparameters.html
opt_params = OptimizationHyperparameters(opttol=1e-4)

# Note: 'newton' is usually faster than BFGS, both are prety fast with so few constraints.
result = QCQP.solve_current_dual_problem(method = 'newton', init_lags = lags_init, opt_params = opt_params)
print(f"bound: {result[0]}, time: {time.time()-t1}s, Lagrange multipliers: {QCQP.current_lags}")

Precomputed 2 A matrices and Fs vectors.
bound: 0.0007481142715895486, time: 3.8845527172088623s, Lagrange multipliers: [-0.01168043  0.00307963]


In [128]:
# A is the total system matrix. Here I'm checking the minimum eigenvalue to see how PSD it is.
lags = QCQP.current_lags
totalA = QCQP._get_total_A(lags)
eigenvals, eigenvecs = np.linalg.eig(totalA)
print(np.min(eigenvals))

np.savetxt("lags_optimal.txt", QCQP.current_lags)
np.savetxt("Pdiags_optimal_real.txt", np.real(QCQP.Proj.Pdiags))
np.savetxt("Pdiags_optimal_imag.txt", np.imag(QCQP.Proj.Pdiags))

(0.03428465935717894+9.34924275127891e-30j)


In [ ]:
# print constraint value, for global constraints it should be satisfied
# Usually lower opttol + newton will mean better convergence
xOpt = QCQP.current_xstar
qTerm = - xOpt.conj().T @  Umat.conj().T @ xOpt
lTerm = 2*xOpt.conj().T @ eVec
print(f"constraint quadratic: {qTerm}")
print(f"constraint linear: {lTerm}")
# print(qTerm + lTerm)

err = np.abs(np.imag(qTerm + lTerm)/np.imag(qTerm))
print(f"power relative error: {err}")

Ivec = xOpt # optimal current vector

constraint quadratic: (0.11977112174086699+0.03157852099847849j)
constraint linear: (-0.11977112838063801-0.0315785245024084j)
power relative error: 1.1095927862733016e-07


In [130]:
scateredPower = 0.5*np.real(Ivec.conj().T @ R0 @ Ivec)
absorbedPower = 0.5*np.real(Ivec.conj().T @ Rmat @ Ivec)
exttincedPower = scateredPower + absorbedPower

# print quantities
print(f"Absorbed power: {absorbedPower} W") 
print(f"Scattered power: {scateredPower} W")
print(f"Extincted power: {exttincedPower} W")

Ivec = Mfactor @ Ivec # optimal current vector in original space

# R0 = np.loadtxt(r"R0_matrix.txt", delimiter=',') # radiated power matrix
# scateredPower = 0.5*np.real(Ivec.conj().T @ R0 @ Ivec)
# print(f"Scattered power: {scateredPower} W")

# save optimal current complex vector as a real-imag text file
Ivec_realimag = np.zeros((Ndes,2))  # create a new array with two columns
Ivec_realimag[:,0] = np.real(Ivec)  # first column is the real part
Ivec_realimag[:,1] = np.imag(Ivec)  # second column is the imaginary part
np.savetxt("Ivec_optimal_realimag.txt", Ivec_realimag, delimiter=',')



Absorbed power: 0.015041146315995886 W
Scattered power: 0.0007481141832433547 W
Extincted power: 0.01578926049923924 W


In [131]:
from dolphindes.cvxopt.gcd import GCDHyperparameters
import copy
gcd_QCQP = copy.deepcopy(QCQP)

t = time.time()
gcd_tol = 1e-2
gcd_params = GCDHyperparameters(gcd_tol=gcd_tol)
gcd_QCQP.run_gcd(gcd_params=gcd_params)
print(f'gcd with gcd_tol={gcd_tol} took time {time.time()-t}.')

Precomputed 2 A matrices and Fs vectors.
At GCD iteration #1, best dual bound found is             0.000748114271589547.
At GCD iteration #2, best dual bound found is             0.0007327296514477254.
At GCD iteration #3, best dual bound found is             0.0007312181576601406.
At GCD iteration #4, best dual bound found is             0.0007290578717244931.
At GCD iteration #5, best dual bound found is             0.0007287194157977084.
At GCD iteration #6, best dual bound found is             0.0007286317811892571.
At GCD iteration #7, best dual bound found is             0.0007285969467791076.
At GCD iteration #8, best dual bound found is             0.0007270118100733517.
At GCD iteration #9, best dual bound found is             0.0007281924040826146.
At GCD iteration #10, best dual bound found is             0.0007284229404873755.
gcd with gcd_tol=0.01 took time 13.789167881011963.


In [132]:
# Now let's check again for PSD of A. We don't expect the constraints to hold or to have strong duality
lags = gcd_QCQP.current_lags
totalA = gcd_QCQP._get_total_A(lags)
eigenvals, eigenvecs = np.linalg.eig(totalA)
print(f"minimal eigenvalue: {np.min(eigenvals)}")

xOpt = gcd_QCQP.current_xstar
qTerm = - xOpt.conj().T @  Umat.conj().T @ xOpt
lTerm = 2*xOpt.conj().T @ eVec
print(f"constraint quadratic: {qTerm}")
print(f"constraint linear: {lTerm}")
print(f"constraint value: {qTerm + lTerm}")

minimal eigenvalue: (0.01570621615033879+2.963205355151616e-17j)
constraint quadratic: (0.10019438495073754+0.019400109026704193j)
constraint linear: (-0.1231244879473515-0.03186479312281728j)
constraint value: (-0.022930102996613952-0.012464684096113086j)


In [133]:
# Now I take the solved gcd_QCQP and finetune it with BFGS if necessary.
t1 = time.time()
gcd_final_result = gcd_QCQP.solve_current_dual_problem(method = 'bfgs', init_lags = gcd_QCQP.current_lags, opt_params = None)
print(f"bound: {gcd_final_result[0]}, time: {time.time()-t1}s")

# Let's check for PSD-ness and constraints again
from matplotlib.pylab import f


lags = gcd_QCQP.current_lags
Pdiags = gcd_QCQP.Proj.Pdiags
totalA = gcd_QCQP._get_total_A(lags)
eigenvals, eigenvecs = np.linalg.eig(totalA)
print(f"min eigenvalue: {np.min(eigenvals)}")

xOpt = gcd_QCQP.current_xstar
qTerm = - xOpt.conj().T @  Umat.conj().T @ xOpt
lTerm = 2*xOpt.conj().T @ eVec
print(f"constraint quadratic: {qTerm}")
print(f"constraint linear: {lTerm}")
print(f"constraint total: {qTerm + lTerm}")


np.savetxt("lags_optimal_GCD.txt", gcd_QCQP.current_lags)
np.savetxt("Pdiags_optimal_GCDreal.txt", np.real(gcd_QCQP.Proj.Pdiags))
np.savetxt("Pdiags_optimal_GCDimag.txt", np.imag(gcd_QCQP.Proj.Pdiags))

bound: 0.0007285596233094809, time: 1.43287992477417s
min eigenvalue: (0.01658511987921709-1.568869541869134e-16j)
constraint quadratic: (0.10011887968236347+0.019385899600228233j)
constraint linear: (-0.12311606938143109-0.031842214140131796j)
constraint total: (-0.022997189699067624-0.012456314539903562j)


In [ ]:
Ivec = xOpt # optimal current vector

scateredPower = 0.5*np.real(Ivec.conj().T @ R0 @ Ivec)
absorbedPower = 0.5*np.real(Ivec.conj().T @ Rmat @ Ivec)
exttincedPower = scateredPower + absorbedPower

# print quantities
print(f"Absorbed power: {absorbedPower} W") 
print(f"Scattered power: {scateredPower} W")
print(f"Extincted power: {exttincedPower} W")

Ivec = Mfactor @ Ivec # optimal current vector in original space

# save optimal current complex vector as a real-imag text file
Ivec_realimag = np.zeros((Ndes,2))  # create a new array with two columns
Ivec_realimag[:,0] = np.real(Ivec)  # first column is the real part
Ivec_realimag[:,1] = np.imag(Ivec)  # second column is the imaginary part
np.savetxt("Ivec_optimal_realimagGCD.txt", Ivec_realimag, delimiter=',')


Absorbed power: 0.00926331674919617 W
Scattered power: 0.00042963305091793205 W
Extincted power: 0.009692949800114101 W


In [ ]:
import matplotlib.pyplot as plt

# 1. Define the range of factors to sweep
mRed = 8 # number of decades to reduce conductivity by
factors = np.logspace(0, -mRed, 25)
min_eigenvalues = []
dual_bounds = []

# Constants assumed from previous cells:
# omega, epsilon_0, c, Lmat, R0, X0, Vinc, Bmat, bVec, beta, eVec, Vzero, Pdiags, X0Norm

print(f"Starting sweep over {len(factors)} points...")

for i, factor in enumerate(factors):
    # --- A. Update Physics based on Factor ---
    copper_conductivity = factor * 5.96e7 # Adjusted conductivity
    copper_permittivity = -1j * copper_conductivity / (omega * epsilon_0)
    
    # Recalculate Surface Impedance (Zs)
    delta = np.sqrt(2 / (omega * mu_0 * copper_conductivity)) # skin depth in m
    Zs = 1.0 / (copper_conductivity * delta)
    Rmat = Zs * Lmat # updated resistance matrix
    
    # --- B. Update Matrices ---
    # Dissipation matrix changes because Zs changes
    Rmat0 = R0 + Rmat # dissipation matrix  
    
    # Impedance matrix Zmat and Umat
    Zmat = Rmat0 + 1j * X0

    Umat = 1j * Zmat.conj() # Dolphindes U matrix
    eVec = -1j*Vinc.conj() / 2 # Dolphindes e vector
    Bmat = -0.5*R0 # quadratic objective matrix
    bVec = Vzero/2 # linear objective vector
    beta = 0 # constant objective term 
 
    # Umat = 1/Zs*1j * Zmat.conj() # Dolphindes U matrix
    # eVec = 1/np.sqrt(Zs)*(-1j*Vinc.conj() / 2) # Dolphindes e vector
    # Bmat =  1/Zs*(-0.5*R0) # quadratic objective matrix
    # bVec = 1/np.sqrt(Zs)*Vzero/2 # linear objective vector
    # beta = 0 # constant objective term
    
    # Re-normalize constraints
    norm_list = [1, 1]
    Plist = [sp.diags(Pdiags[:, k] / norm_list[k]) for k in range(Pdiags.shape[1])]

    # --- C. Setup QCQP ---
    # We create a new QCQP instance for this specific conductivity
    # verbose=0 to keep the output clean during the loop
    gcdIter_QCQP = DenseSharedProjQCQP(Bmat, bVec, beta,
                               Umat, eVec,
                               Plist, verbose=0)

    # --- D. Run GCD ---
    gcd_tol = 1e-2
    gcd_params = GCDHyperparameters(gcd_tol=gcd_tol)
    
    # Run the solver
    gcdIter_QCQP.solve_current_dual_problem(method = 'newton', init_lags = lags_init, opt_params = opt_params)

    gcdIter_QCQP.run_gcd(gcd_params=gcd_params)

    gcdIter_QCQP.solve_current_dual_problem(method = 'bfgs', init_lags = gcdIter_QCQP.current_lags, opt_params = None)
    
    # --- E. Calculate Minimal Eigenvalue ---
    lags = gcdIter_QCQP.current_lags
    totalA = gcdIter_QCQP._get_total_A(lags)
    
    # Calculate eigenvalues
    # Note: Using eigvalsh usually assumes Hermitian, which is generally true for the dual matrix
    # but we stick to np.linalg.eig as per your notebook for safety
    eigenvals, _ = np.linalg.eig(totalA)
    
    # We take the real part of the minimum eigenvalue
    min_eig = np.min(np.real(eigenvals))
    min_eigenvalues.append(min_eig)
    dual_bounds.append(gcdIter_QCQP.current_dual)

    
    print(f"Iter {i+1}/{len(factors)}: Factor={factor:.2e}, Min Eig={min_eig:.2e}")


In [ ]:
plt.figure(figsize=(10, 6))
plt.loglog(factors, min_eigenvalues, 'b-o', linewidth=2)
plt.grid(True, which="both", ls="-")
plt.xlabel('Conductivity Reduction Factor "s"')
plt.ylabel('Min Eigenvalue')
plt.title('GCD Minimal Eigenvalue vs Conductivity Reduction')
# plt.axhline(0, color='r', linestyle='--', label='PSD Limit')
# plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.loglog(factors, np.abs(dual_bounds), 'r-s', linewidth=2)
plt.grid(True, which="both", ls="-")
plt.xlabel('Conductivity Reduction Factor "s"')
plt.ylabel('Dual Bound')
plt.title('GCD Dual Bound vs Conductivity Reduction')
plt.show()